## Depression Detection (Part-2): Reddit Mental Health Dataset Pipeline

This notebook builds an end-to-end, modular pipeline for **cross-domain comparison** between:
- **DAIC-WOZ clinical interviews (Part-1)**
- **Reddit social media posts (Part-2)**

### What you’ll get (saved under `outputs/`)
- `reddit_embeddings.npy`
- `reddit_labels.npy`
- `results_comparison.csv`
- Multiple `.png` visualizations (class balance, confusion matrices, F1 comparison, length histogram)

### Notes
- Designed to run **top-to-bottom on Google Colab**.
- All randomness uses `random_state=42`.
- The pipeline expects the Kaggle dataset format (CSV(s) containing at least `subreddit` and a text column like `selftext` or `body`).

In [ ]:
# If running on Google Colab, uncomment the next line.
# !pip -q install -U pandas numpy scikit-learn sentence-transformers matplotlib seaborn

# Quick dependency check (prints what is missing)
missing = []
try:
    import pandas as pd  # noqa: F401
except Exception:
    missing.append("pandas")
try:
    import numpy as np  # noqa: F401
except Exception:
    missing.append("numpy")
try:
    import sklearn  # noqa: F401
except Exception:
    missing.append("scikit-learn")
try:
    import sentence_transformers  # noqa: F401
except Exception:
    missing.append("sentence-transformers")
try:
    import matplotlib  # noqa: F401
except Exception:
    missing.append("matplotlib")
try:
    import seaborn as sns  # noqa: F401
except Exception:
    missing.append("seaborn")

if missing:
    print("Missing packages:", missing)
    print("If on Colab: uncomment the pip install line above and re-run.")
else:
    print("All core packages are importable.")

In [ ]:
import os
import re
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

print("Outputs will be saved to:", OUTPUT_DIR.resolve())

## Step 1: Load & Explore

We load one or many CSV files from the Reddit Mental Health dataset, then print:
- shape, column names, sample rows
- subreddit value counts
- null counts for all columns

This notebook supports:
- a single CSV file path
- a directory (recursively loads all `*.csv` under it)
- an explicit list of CSV file paths

In [ ]:
def load_reddit_csvs(
    data_path: str,
    file_glob: str = "**/*.csv",
    max_files: Optional[int] = None,
    encoding: Optional[str] = None,
) -> Tuple[pd.DataFrame, List[Path]]:
    """Load one CSV, many CSVs from a directory, or a glob.

    Args:
        data_path: file or directory path.
        file_glob: used if data_path is a directory.
        max_files: optionally cap number of CSVs (useful for quick tests).
        encoding: optional encoding passed to pandas.

    Returns:
        (df, files_loaded)
    """
    p = Path(data_path)
    if not p.exists():
        raise FileNotFoundError(
            f"DATA_PATH does not exist: {data_path}. "
            "Set DATA_PATH to your Kaggle CSV file, or to a directory containing CSVs."
        )

    if p.is_file():
        files = [p]
    else:
        files = sorted(p.glob(file_glob))
        files = [f for f in files if f.is_file() and f.suffix.lower() == ".csv"]

    if not files:
        raise FileNotFoundError(
            f"No CSV files found under: {p} (glob: {file_glob})."
        )

    if max_files is not None:
        files = files[:max_files]

    dfs = []
    for f in files:
        try:
            dfs.append(pd.read_csv(f, encoding=encoding))
        except UnicodeDecodeError:
            dfs.append(pd.read_csv(f, encoding=encoding or "latin-1"))
        except Exception as e:
            raise RuntimeError(f"Failed to read {f}: {e}")

    df = pd.concat(dfs, ignore_index=True)
    return df, files


def explore_dataframe(df: pd.DataFrame, subreddit_col: str = "subreddit", n: int = 5) -> None:
    print("Shape:", df.shape)
    print("\nColumns:")
    print(list(df.columns))

    print("\nSample rows:")
    display(df.head(n))

    if subreddit_col in df.columns:
        print("\nSubreddit value counts (top 30):")
        display(df[subreddit_col].value_counts(dropna=False).head(30))
    else:
        print(f"\nColumn '{subreddit_col}' not found; cannot compute subreddit counts.")

    print("\nNull counts:")
    display(df.isna().sum().sort_values(ascending=False).head(50))


# --- Set your data location here ---
# You can point to a single CSV, or to a directory containing CSVs (loaded recursively).
#
# If you run on Colab, set DATA_PATH to where you uploaded/unzipped the Kaggle dataset.
# Examples:
# DATA_PATH = "/content/reddit-mental-health-dataset"  # directory
# DATA_PATH = "/content/drive/MyDrive/reddit-mental-health-dataset"  # directory
# DATA_PATH = "/content/some_file.csv"  # single file
#
# Local example (this repo layout):
# DATA_PATH = r"Original Reddit Data/raw data"

CANDIDATE_DATA_PATHS = [
    r"Original Reddit Data/raw data",
    "/content/reddit-mental-health-dataset",
    "/content/reddit_dataset",
    "/content/dataset",
    "/content/drive/MyDrive/reddit-mental-health-dataset",
]

DATA_PATH = next((p for p in CANDIDATE_DATA_PATHS if Path(p).exists()), None)
if DATA_PATH is None:
    # Force an explicit, helpful error rather than failing deeper in the pipeline.
    raise FileNotFoundError(
        "Could not auto-detect the dataset location. Set DATA_PATH to your Reddit CSV file "
        "or to a directory containing the Kaggle CSVs, then re-run this cell."
    )

print("Using DATA_PATH:", DATA_PATH)
reddit_df_raw, files_loaded = load_reddit_csvs(DATA_PATH)
print(f"Loaded {len(files_loaded)} file(s).")
explore_dataframe(reddit_df_raw)

## Step 2: Label Engineering

We map specific subreddits into **3 risk categories**:
- **Label 2 (High Risk)**: `suicidewatch`
- **Label 1 (Mental Health Risk)**: `depression`, `anxiety`, `bipolarreddit`, `ptsd`, `schizophrenia`, `bpd`, `socialanxiety`, `healthanxiety`, `lonely`
- **Label 0 (Control)**: `fitness`, `jokes`, `parenting`, `personalfinance`, `relationships`, `meditation`, `teaching`, `legaladvice`

Rows whose subreddit is not in this mapping are dropped.

Outputs added:
- `risk_label` ∈ {0,1,2}
- `risk_label_name` ∈ {Control, Mental Health Risk, High Risk}

In [ ]:
RISK_MAPPING: Dict[str, int] = {
    # High risk
    "suicidewatch": 2,

    # Mental health risk
    "depression": 1,
    "anxiety": 1,
    "bipolarreddit": 1,
    "ptsd": 1,
    "schizophrenia": 1,
    "bpd": 1,
    "socialanxiety": 1,
    "healthanxiety": 1,
    "lonely": 1,

    # Control
    "fitness": 0,
    "jokes": 0,
    "parenting": 0,
    "personalfinance": 0,
    "relationships": 0,
    "meditation": 0,
    "teaching": 0,
    "legaladvice": 0,
}

LABEL_NAME = {
    0: "Control",
    1: "Mental Health Risk",
    2: "High Risk",
}


def apply_risk_labels(df: pd.DataFrame, subreddit_col: str = "subreddit") -> pd.DataFrame:
    if subreddit_col not in df.columns:
        raise KeyError(
            f"Expected a '{subreddit_col}' column but it was not found. "
            f"Available columns: {list(df.columns)}"
        )

    out = df.copy()
    out[subreddit_col] = out[subreddit_col].astype(str).str.strip().str.lower()

    out["risk_label"] = out[subreddit_col].map(RISK_MAPPING)
    out = out.dropna(subset=["risk_label"]).copy()
    out["risk_label"] = out["risk_label"].astype(int)
    out["risk_label_name"] = out["risk_label"].map(LABEL_NAME)

    return out


reddit_df_labeled = apply_risk_labels(reddit_df_raw)

print("After labeling + dropping unmapped subreddits:")
print("Shape:", reddit_df_labeled.shape)
print("\nClass distribution:")
display(reddit_df_labeled["risk_label_name"].value_counts())

present = set(reddit_df_labeled["risk_label"].unique().tolist())
missing = {0, 1, 2} - present
if missing:
    raise ValueError(
        "After applying the required subreddit-to-label mapping, "
        f"the dataset is missing label(s): {sorted(missing)}.\n"
        f"Present labels: {sorted(present)}.\n"
        "This usually means you haven't loaded CSVs for one or more mapped subreddits (especially Control).\n"
        "Fix: update DATA_PATH to include those subreddits' CSV files, then re-run from Step 1."
    )

## Step 3: Text Preprocessing

We clean the text (`selftext` or `body`) with the following rules:
- Drop rows where text is null, `[deleted]`, `[removed]`, or empty
- Lowercase
- Remove URLs (`http/https`)
- Remove subreddit mentions (`r/word`) and user mentions (`u/word`)
- Remove special characters, keeping only alphanumeric + basic punctuation
- Strip extra whitespace
- Drop posts with fewer than **20 words** after cleaning
- Deduplicate by **author** (keep only the **most recent** post per author) to reduce leakage
- Undersample to **2000 posts per class** (balanced), `random_state=42`

We also track and plot:
- class distribution **before** and **after** balancing
- word count distribution by risk label

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)
MENTION_RE = re.compile(r"\b(?:r|u)/[A-Za-z0-9_]+\b", flags=re.IGNORECASE)
# Keep: a-z0-9 whitespace + basic punctuation
ALLOWED_CHARS_RE = re.compile(r"[^a-z0-9\s\.,!?;:'\"\-\(\)]")
WHITESPACE_RE = re.compile(r"\s+")


def pick_text_column(df: pd.DataFrame, preferred: Iterable[str] = ("selftext", "body")) -> str:
    for c in preferred:
        if c in df.columns:
            return c
    raise KeyError(
        f"Could not find any expected text column {list(preferred)}. "
        f"Available columns: {list(df.columns)}"
    )


def clean_text(text: str) -> str:
    if text is None:
        return ""
    t = str(text).strip()
    if not t:
        return ""

    lowered = t.lower()
    if lowered in {"[deleted]", "[removed]"}:
        return ""

    lowered = URL_RE.sub(" ", lowered)
    lowered = MENTION_RE.sub(" ", lowered)
    lowered = ALLOWED_CHARS_RE.sub(" ", lowered)
    lowered = WHITESPACE_RE.sub(" ", lowered).strip()
    return lowered


def word_count(text: str) -> int:
    if not text:
        return 0
    return len(text.split())


def dedupe_by_author_most_recent(
    df: pd.DataFrame,
    author_col: str = "author",
    created_utc_col: str = "created_utc",
    fallback_time_col: str = "timestamp",
) -> pd.DataFrame:
    if author_col not in df.columns:
        print(f"Warning: '{author_col}' column not found; skipping author deduplication.")
        return df

    out = df.copy()
    if created_utc_col in out.columns:
        out[created_utc_col] = pd.to_numeric(out[created_utc_col], errors="coerce")
        sort_col = created_utc_col
    elif fallback_time_col in out.columns:
        out[fallback_time_col] = pd.to_datetime(out[fallback_time_col], errors="coerce")
        sort_col = fallback_time_col
    else:
        print(
            f"Warning: no '{created_utc_col}' or '{fallback_time_col}' column found; "
            "dedup will keep the last row per author by original order."
        )
        out["__row_id"] = np.arange(len(out))
        sort_col = "__row_id"

    out = out.sort_values(sort_col, ascending=True)
    out = out.drop_duplicates(subset=[author_col], keep="last")

    if "__row_id" in out.columns:
        out = out.drop(columns=["__row_id"])

    return out


def balance_undersample(df: pd.DataFrame, label_col: str = "risk_label", target_per_class: int = 2000) -> pd.DataFrame:
    counts = df[label_col].value_counts().sort_index()
    min_count = int(counts.min())

    if min_count == 0:
        raise ValueError("At least one class has 0 samples after preprocessing.")

    effective_target = min(target_per_class, min_count)
    if effective_target < target_per_class:
        print(
            f"Warning: At least one class has fewer than {target_per_class} samples. "
            f"Balancing to {effective_target} per class instead."
        )

    balanced = (
        df.groupby(label_col, group_keys=False)
        .apply(lambda g: g.sample(n=effective_target, random_state=RANDOM_STATE, replace=False))
        .reset_index(drop=True)
    )
    return balanced


TEXT_COL = pick_text_column(reddit_df_labeled)
print("Using text column:", TEXT_COL)

reddit_df = reddit_df_labeled.copy()
reddit_df["text_raw"] = reddit_df[TEXT_COL]

# Drop null/empty/deleted/removed
reddit_df["text_clean"] = reddit_df["text_raw"].apply(clean_text)
reddit_df = reddit_df[reddit_df["text_clean"].astype(bool)].copy()

# Drop too-short posts
reddit_df["word_count"] = reddit_df["text_clean"].apply(word_count)
reddit_df = reddit_df[reddit_df["word_count"] >= 20].copy()

# Deduplicate by author (most recent post)
reddit_df = dedupe_by_author_most_recent(reddit_df)

# Class distribution before balancing (after cleaning + dedup)
class_counts_before = reddit_df["risk_label_name"].value_counts().reindex(["Control", "Mental Health Risk", "High Risk"], fill_value=0)
print("Class distribution (after cleaning + dedup, before balancing):")
display(class_counts_before)
print("Dataset size:", len(reddit_df))

# Balance classes
reddit_df_balanced = balance_undersample(reddit_df, target_per_class=2000)
class_counts_after = reddit_df_balanced["risk_label_name"].value_counts().reindex(["Control", "Mental Health Risk", "High Risk"], fill_value=0)

print("\nFinal class distribution (after balancing):")
display(class_counts_after)
print("Final dataset size:", len(reddit_df_balanced))

# Safety checks
assert reddit_df_balanced["text_clean"].isna().sum() == 0
assert reddit_df_balanced["text_clean"].str.len().min() > 0

## Step 4: Generate MiniLM Embeddings

We use `sentence-transformers` with **`all-MiniLM-L6-v2`** (same as Part-1) to generate one embedding per post.

We save:
- `outputs/reddit_embeddings.npy`
- `outputs/reddit_labels.npy`

In [ ]:
from sentence_transformers import SentenceTransformer


def generate_embeddings(
    texts: List[str],
    model_name: str = "all-MiniLM-L6-v2",
    batch_size: int = 64,
    normalize_embeddings: bool = False,
) -> np.ndarray:
    """Generate sentence embeddings for a list of texts."""
    try:
        import torch

        device = "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        device = "cpu"

    model = SentenceTransformer(model_name, device=device)
    emb = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=normalize_embeddings,
    )
    return emb.astype(np.float32)


texts = reddit_df_balanced["text_clean"].tolist()
labels = reddit_df_balanced["risk_label"].to_numpy(dtype=np.int64)

embeddings = generate_embeddings(texts)

emb_path = OUTPUT_DIR / "reddit_embeddings.npy"
lab_path = OUTPUT_DIR / "reddit_labels.npy"
np.save(emb_path, embeddings)
np.save(lab_path, labels)

print("Embedding matrix shape:", embeddings.shape)
print("Labels shape:", labels.shape)
print("Saved:", emb_path, "and", lab_path)

## Step 5: Classical Model Training & Evaluation (Stratified 5-Fold CV)

Because Reddit posts do not have a participant/interview structure like DAIC-WOZ, we use **stratified 5-fold cross validation**.

Models:
1. Logistic Regression (`max_iter=1000`, `class_weight=balanced`)
2. SVM (RBF kernel, `class_weight=balanced`)
3. MLP (`hidden_layer_sizes=(256, 128)`, `max_iter=500`)

For each model and fold, we collect:
- Accuracy
- Precision / Recall / F1 (weighted + macro)
- Confusion matrix

Then we report mean±std across folds, plus an aggregated confusion matrix.

In [ ]:
def build_models() -> Dict[str, Pipeline]:
    return {
        "LR": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=1000,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "SVM": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "clf",
                    SVC(
                        kernel="rbf",
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "MLP": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "clf",
                    MLPClassifier(
                        hidden_layer_sizes=(256, 128),
                        max_iter=500,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
    }


def evaluate_models_cv(
    X: np.ndarray,
    y: np.ndarray,
    model_dict: Dict[str, Pipeline],
    n_splits: int = 5,
) -> Tuple[pd.DataFrame, Dict[str, np.ndarray], Dict[str, np.ndarray]]:
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    rows = []
    cm_sum_by_model: Dict[str, np.ndarray] = {}
    cm_avg_by_model: Dict[str, np.ndarray] = {}

    labels_sorted = np.array(sorted(np.unique(y)))
    n_classes = len(labels_sorted)

    for model_name, model in model_dict.items():
        cm_sum = np.zeros((n_classes, n_classes), dtype=np.int64)

        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

            acc = accuracy_score(y_test, y_pred)
            p_w, r_w, f1_w, _ = precision_recall_fscore_support(
                y_test, y_pred, average="weighted", zero_division=0
            )
            p_m, r_m, f1_m, _ = precision_recall_fscore_support(
                y_test, y_pred, average="macro", zero_division=0
            )

            cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)
            cm_sum += cm

            rows.append(
                {
                    "model": model_name,
                    "fold": fold_idx,
                    "accuracy": acc,
                    "precision_weighted": p_w,
                    "recall_weighted": r_w,
                    "f1_weighted": f1_w,
                    "precision_macro": p_m,
                    "recall_macro": r_m,
                    "f1_macro": f1_m,
                }
            )

        cm_sum_by_model[model_name] = cm_sum
        cm_avg_by_model[model_name] = cm_sum / n_splits

    metrics_df = pd.DataFrame(rows)
    return metrics_df, cm_sum_by_model, cm_avg_by_model


X = embeddings
y = labels
models = build_models()

metrics_df, cm_sum_by_model, cm_avg_by_model = evaluate_models_cv(X, y, models, n_splits=5)

# Per-fold metrics
print("Per-fold metrics (first 10 rows):")
display(metrics_df.head(10))

# Mean ± std summary
summary = (
    metrics_df.groupby("model")
    .agg(["mean", "std"])
    .sort_index()
)

print("\nMean ± std across folds:")
display(summary)

# Aggregated confusion matrices (sum across folds)
print("\nAggregated confusion matrices (sum across folds):")
for model_name, cm_sum in cm_sum_by_model.items():
    print("\nModel:", model_name)
    display(pd.DataFrame(cm_sum, index=[0, 1, 2], columns=[0, 1, 2]))

# Clean results table (includes accuracy, precision, recall, F1 — macro & weighted)
results_table = (
    metrics_df.groupby("model")
    .agg(
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        precision_weighted_mean=("precision_weighted", "mean"),
        precision_weighted_std=("precision_weighted", "std"),
        recall_weighted_mean=("recall_weighted", "mean"),
        recall_weighted_std=("recall_weighted", "std"),
        f1_weighted_mean=("f1_weighted", "mean"),
        f1_weighted_std=("f1_weighted", "std"),
        precision_macro_mean=("precision_macro", "mean"),
        precision_macro_std=("precision_macro", "std"),
        recall_macro_mean=("recall_macro", "mean"),
        recall_macro_std=("recall_macro", "std"),
        f1_macro_mean=("f1_macro", "mean"),
        f1_macro_std=("f1_macro", "std"),
    )
    .reset_index()
    .sort_values("accuracy_mean", ascending=False)
)

print("\nResults summary table:")
display(results_table)

# Extract Reddit metrics for Step 6
reddit_model_results = results_table.set_index("model")[["accuracy_mean", "f1_weighted_mean", "f1_macro_mean"]].copy()

## Step 6: Results Comparison Table (DAIC-WOZ vs Reddit)

We create a comparison table mirroring Part-1 format:

| Model | Dataset | Accuracy | Weighted F1 | Macro F1 |
|-------|---------|----------|-------------|----------|

We pre-fill DAIC-WOZ results (static), then append the Reddit cross-validation results from Step 5.

The table is printed and saved as `outputs/results_comparison.csv`.

In [ ]:
DAIC_STATIC_ROWS = [
    {"Model": "LR", "Dataset": "DAIC-WOZ", "Accuracy": 0.65, "Weighted F1": 0.51, "Macro F1": 0.39},
    {"Model": "SVM/QSVC", "Dataset": "DAIC-WOZ", "Accuracy": 0.47, "Weighted F1": 0.44, "Macro F1": 0.44},
    {"Model": "QSVM", "Dataset": "DAIC-WOZ", "Accuracy": 0.47, "Weighted F1": 0.44, "Macro F1": 0.44},
]

reddit_rows = []
for model_name in ["LR", "SVM", "MLP"]:
    if model_name not in reddit_model_results.index:
        continue
    reddit_rows.append(
        {
            "Model": model_name,
            "Dataset": "Reddit",
            "Accuracy": float(reddit_model_results.loc[model_name, "accuracy_mean"]),
            "Weighted F1": float(reddit_model_results.loc[model_name, "f1_weighted_mean"]),
            "Macro F1": float(reddit_model_results.loc[model_name, "f1_macro_mean"]),
        }
    )

comparison_df = pd.DataFrame(DAIC_STATIC_ROWS + reddit_rows)

print("Results comparison (DAIC-WOZ vs Reddit):")
display(comparison_df)

comparison_path = OUTPUT_DIR / "results_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print("Saved:", comparison_path.resolve())

## Step 7: Visualizations

We generate and save the following plots (PNG) under `outputs/`:

1. Class distribution bar chart (before and after balancing)
2. Confusion matrix heatmaps for each model (averaged across folds)
3. F1-score comparison bar chart: DAIC-WOZ vs Reddit, grouped by model
4. Post length distribution histogram (word count) by risk label

In [ ]:
def save_fig(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()


# 1) Class distribution (before vs after balancing)
fig, ax = plt.subplots(figsize=(8, 4))
dist_df = pd.DataFrame(
    {
        "Before balancing": class_counts_before,
        "After balancing": class_counts_after,
    }
).reset_index(names="risk_label_name")

melted = dist_df.melt(id_vars="risk_label_name", var_name="stage", value_name="count")
sns.barplot(data=melted, x="risk_label_name", y="count", hue="stage", ax=ax)
ax.set_title("Class distribution before vs after balancing")
ax.set_xlabel("Risk label")
ax.set_ylabel("Post count")
ax.legend(title="")

save_fig(OUTPUT_DIR / "class_distribution_before_after.png")


# Helper for confusion matrices
LABEL_ORDER = [0, 1, 2]
LABEL_NAMES = [LABEL_NAME[i] for i in LABEL_ORDER]


def plot_confusion_matrix_heatmap(cm: np.ndarray, title: str, out_path: Path, normalize: bool = False) -> None:
    cm_plot = cm.astype(float)
    if normalize:
        row_sums = cm_plot.sum(axis=1, keepdims=True)
        cm_plot = np.divide(cm_plot, row_sums, out=np.zeros_like(cm_plot), where=row_sums != 0)

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    fmt = ".2f" if normalize else ".0f"
    sns.heatmap(
        cm_plot,
        annot=True,
        fmt=fmt,
        cmap="Blues",
        xticklabels=LABEL_NAMES,
        yticklabels=LABEL_NAMES,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    save_fig(out_path)


# 2) Confusion matrices (averaged across folds)
for model_name, cm_avg in cm_avg_by_model.items():
    plot_confusion_matrix_heatmap(
        cm_avg,
        title=f"{model_name} confusion matrix (avg counts across folds)",
        out_path=OUTPUT_DIR / f"cm_{model_name}_avg_counts.png",
        normalize=False,
    )
    plot_confusion_matrix_heatmap(
        cm_avg,
        title=f"{model_name} confusion matrix (row-normalized)",
        out_path=OUTPUT_DIR / f"cm_{model_name}_row_normalized.png",
        normalize=True,
    )


# 3) F1 comparison: DAIC-WOZ vs Reddit
# Use only rows that exist (some models may not have both datasets).
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = comparison_df.copy()

# More compact labels for plotting
plot_df["Model_plot"] = plot_df["Model"].replace({"SVM/QSVC": "SVM"})

sns.barplot(data=plot_df, x="Model_plot", y="Weighted F1", hue="Dataset", ax=ax)
ax.set_title("Weighted F1: DAIC-WOZ vs Reddit")
ax.set_xlabel("Model")
ax.set_ylabel("Weighted F1")
ax.legend(title="Dataset")

save_fig(OUTPUT_DIR / "weighted_f1_daic_vs_reddit.png")


# 4) Post length distribution histogram by risk label (after cleaning + dedup, before balancing)
fig, ax = plt.subplots(figsize=(9, 4.5))
length_df = reddit_df.copy()
length_df["risk_label_name"] = pd.Categorical(
    length_df["risk_label_name"],
    categories=["Control", "Mental Health Risk", "High Risk"],
    ordered=True,
)

sns.histplot(
    data=length_df,
    x="word_count",
    hue="risk_label_name",
    bins=50,
    element="step",
    stat="density",
    common_norm=False,
    ax=ax,
)
ax.set_title("Post length distribution (word count) by risk label")
ax.set_xlabel("Word count")
ax.set_ylabel("Density")
ax.set_xlim(0, np.percentile(length_df["word_count"], 99))

save_fig(OUTPUT_DIR / "post_length_by_label.png")

print("Saved plots to:", OUTPUT_DIR.resolve())

## Final Printed Summary

We print a compact summary so your report can clearly state the key finding: **how Reddit social media results compare to DAIC-WOZ clinical interview results**.

In [ ]:
def format_mean_std(mean_val: float, std_val: float) -> str:
    if pd.isna(std_val):
        return f"{mean_val:.3f}"
    return f"{mean_val:.3f} ± {std_val:.3f}"


print("=== Reddit (Part-2) CV Results (mean ± std) ===")
for _, row in results_table.sort_values("model").iterrows():
    print(
        f"{row['model']}: "
        f"Acc={format_mean_std(row['accuracy_mean'], row['accuracy_std'])}, "
        f"Weighted F1={format_mean_std(row['f1_weighted_mean'], row['f1_weighted_std'])}, "
        f"Macro F1={format_mean_std(row['f1_macro_mean'], row['f1_macro_std'])}"
    )

print("\n=== Cross-domain comparison table ===")
display(comparison_df)

# Simple key finding statement focusing on Weighted F1 and Accuracy
reddit_best = results_table.sort_values("f1_weighted_mean", ascending=False).iloc[0]
print(
    "\nKey finding (auto-generated): "
    f"On Reddit, the best Weighted F1 among classical models is {reddit_best['f1_weighted_mean']:.3f} "
    f"({reddit_best['model']}). Compare this against DAIC-WOZ baselines in the table above."
)

print("\nArtifacts written to:", OUTPUT_DIR.resolve())
print("- reddit_embeddings.npy")
print("- reddit_labels.npy")
print("- results_comparison.csv")
print("- plots (*.png)")

## (Extension) Step 8: DAIC-WOZ Loading + Cross-Domain Transfer (Classical-only, Optional)

This section upgrades the work to **major-project level** by measuring **cross-domain generalization**:

- Train on **DAIC-WOZ** → test on **Reddit**
- Train on **Reddit** → test on **DAIC-WOZ**

### Why this matters
Within-dataset CV answers: *"Can we classify examples from the same domain?"*

Cross-domain transfer answers a harder question: *"Does a model learned from clinical interviews generalize to real-world social media language (and vice versa)?"*

### What you need
- DAIC-WOZ transcript folder containing `*_TRANSCRIPT.csv`
- DAIC-WOZ label CSV (this notebook supports the common `avec_combined_labels.csv` format with `Participant_ID`, `PHQ8_Binary`, `PHQ8_Score`)

If the DAIC paths are not available, the notebook will **skip** transfer evaluation gracefully.

In [ ]:
def _first_existing_path(candidates: List[str]) -> Optional[Path]:
    for c in candidates:
        p = Path(c)
        if p.exists():
            return p
    return None


# --- Configure DAIC-WOZ paths here ---
# Local repo: you already have labels under `major 1/avec_combined_labels (1).csv`
# For Colab: point these to where you uploaded/unzipped DAIC-WOZ.
DAIC_LABEL_CANDIDATES = [
    r"major 1/avec_combined_labels (1).csv",
    r"major 1/avec_combined_labels.csv",
    "/content/avec_combined_labels.csv",
    "/content/drive/MyDrive/avec_combined_labels.csv",
]

DAIC_TRANSCRIPT_DIR_CANDIDATES = [
    r"major 1/transcript",
    r"major 1/transcripts",
    "/content/transcript",
    "/content/transcripts",
    "/content/drive/MyDrive/transcript",
    "/content/drive/MyDrive/transcripts",
]

DAIC_LABEL_PATH = _first_existing_path(DAIC_LABEL_CANDIDATES)
DAIC_TRANSCRIPT_DIR = _first_existing_path(DAIC_TRANSCRIPT_DIR_CANDIDATES)

print("DAIC label path:", DAIC_LABEL_PATH)
print("DAIC transcript dir:", DAIC_TRANSCRIPT_DIR)


def load_daic_woz(
    transcript_dir: Path,
    label_csv: Path,
    participant_only: bool = True,
) -> pd.DataFrame:
    """Load DAIC-WOZ transcripts and merge with labels.

    Expects transcript files named like `XXX_TRANSCRIPT.csv` with columns including
    `speaker` and `value` (tab- or comma-separated). Only participant utterances are used.
    """
    if transcript_dir is None or not transcript_dir.exists():
        raise FileNotFoundError(
            "DAIC transcript directory not found. Set DAIC_TRANSCRIPT_DIR to the folder containing *_TRANSCRIPT.csv files."
        )
    if label_csv is None or not label_csv.exists():
        raise FileNotFoundError(
            "DAIC label CSV not found. Set DAIC_LABEL_PATH to your labels file (avec_combined_labels.csv)."
        )

    label_df = pd.read_csv(label_csv)
    label_df.columns = label_df.columns.str.strip()
    required = {"Participant_ID", "PHQ8_Binary", "PHQ8_Score"}
    missing = required - set(label_df.columns)
    if missing:
        raise KeyError(
            f"DAIC label CSV is missing columns: {sorted(missing)}. Found: {list(label_df.columns)}"
        )

    label_df["Participant_ID"] = label_df["Participant_ID"].astype(str)

    rows = []
    files = sorted(list(transcript_dir.glob("*_TRANSCRIPT.csv")))
    if not files:
        raise FileNotFoundError(f"No *_TRANSCRIPT.csv files found under: {transcript_dir}")

    for f in files:
        pid = f.name.split("_")[0]
        try:
            # DAIC transcripts are often tab-separated
            df = pd.read_csv(f, sep="\t")
            if df.shape[1] == 1:
                # fallback to comma
                df = pd.read_csv(f)
        except Exception:
            # last resort: try comma
            df = pd.read_csv(f)

        if not {"speaker", "value"}.issubset(df.columns):
            continue

        if participant_only:
            vals = df.loc[df["speaker"] == "Participant", "value"].dropna().astype(str)
        else:
            vals = df["value"].dropna().astype(str)

        text = " ".join(vals.tolist())
        rows.append({"Participant_ID": str(pid), "transcript": text})

    transcript_df = pd.DataFrame(rows)
    merged = pd.merge(transcript_df, label_df, on="Participant_ID", how="inner")

    if merged.empty:
        raise ValueError(
            "Merged DAIC transcript+labels is empty. Check that transcript filenames match Participant_IDs in the label CSV."
        )

    merged["text_clean"] = merged["transcript"].apply(clean_text)
    merged = merged[merged["text_clean"].astype(bool)].copy()

    # Keep only rows with valid binary labels
    merged["PHQ8_Binary"] = pd.to_numeric(merged["PHQ8_Binary"], errors="coerce")
    merged = merged.dropna(subset=["PHQ8_Binary"]).copy()
    merged["PHQ8_Binary"] = merged["PHQ8_Binary"].astype(int)

    return merged


def evaluate_transfer(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    model_dict: Dict[str, Pipeline],
) -> Tuple[pd.DataFrame, Dict[str, np.ndarray]]:
    labels_sorted = np.array(sorted(np.unique(np.concatenate([y_train, y_test]))))

    rows = []
    cm_by_model: Dict[str, np.ndarray] = {}

    for model_name, model in model_dict.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        p_w, r_w, f1_w, _ = precision_recall_fscore_support(
            y_test, y_pred, average="weighted", zero_division=0
        )
        p_m, r_m, f1_m, _ = precision_recall_fscore_support(
            y_test, y_pred, average="macro", zero_division=0
        )

        cm = confusion_matrix(y_test, y_pred, labels=labels_sorted)
        cm_by_model[model_name] = cm

        rows.append(
            {
                "model": model_name,
                "accuracy": acc,
                "precision_weighted": p_w,
                "recall_weighted": r_w,
                "f1_weighted": f1_w,
                "precision_macro": p_m,
                "recall_macro": r_m,
                "f1_macro": f1_m,
            }
        )

    return pd.DataFrame(rows).sort_values("f1_weighted", ascending=False), cm_by_model


def save_transfer_confusions(
    cm_by_model: Dict[str, np.ndarray],
    title_prefix: str,
    out_prefix: str,
    label_names: List[str],
) -> None:
    for model_name, cm in cm_by_model.items():
        fig, ax = plt.subplots(figsize=(5.5, 4.5))
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=label_names,
            yticklabels=label_names,
            ax=ax,
        )
        ax.set_title(f"{title_prefix} - {model_name}")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        save_fig(OUTPUT_DIR / f"{out_prefix}_{model_name}.png")


# --- Run transfer only if DAIC paths exist ---
if DAIC_LABEL_PATH is None or DAIC_TRANSCRIPT_DIR is None:
    print("\nSkipping DAIC↔Reddit transfer: DAIC paths not found. (This is expected if you haven't added DAIC data to this runtime.)")
else:
    daic_df = load_daic_woz(DAIC_TRANSCRIPT_DIR, DAIC_LABEL_PATH)
    print("Loaded DAIC-WOZ rows:", len(daic_df))
    display(daic_df[["Participant_ID", "PHQ8_Binary", "PHQ8_Score"]].head())

    # Embed DAIC
    daic_embeddings = generate_embeddings(daic_df["text_clean"].tolist())
    daic_labels = daic_df["PHQ8_Binary"].to_numpy(dtype=np.int64)

    np.save(OUTPUT_DIR / "daic_embeddings.npy", daic_embeddings)
    np.save(OUTPUT_DIR / "daic_labels_binary.npy", daic_labels)

    # Create *binary* Reddit labels for cross-domain alignment: Control=0, Any-Risk=1
    reddit_labels_binary = (reddit_df_balanced["risk_label"].to_numpy(dtype=np.int64) > 0).astype(np.int64)

    # Train on DAIC -> test on Reddit
    transfer_models = build_models()

    daic_to_reddit_df, daic_to_reddit_cms = evaluate_transfer(
        daic_embeddings,
        daic_labels,
        embeddings,
        reddit_labels_binary,
        transfer_models,
    )

    # Train on Reddit -> test on DAIC
    reddit_to_daic_df, reddit_to_daic_cms = evaluate_transfer(
        embeddings,
        reddit_labels_binary,
        daic_embeddings,
        daic_labels,
        transfer_models,
    )

    print("\n=== Transfer: Train DAIC-WOZ (binary) → Test Reddit (binary) ===")
    display(daic_to_reddit_df)

    print("\n=== Transfer: Train Reddit (binary) → Test DAIC-WOZ (binary) ===")
    display(reddit_to_daic_df)

    # Save result tables
    daic_to_reddit_df.assign(direction="DAIC→Reddit").to_csv(
        OUTPUT_DIR / "transfer_daic_to_reddit.csv", index=False
    )
    reddit_to_daic_df.assign(direction="Reddit→DAIC").to_csv(
        OUTPUT_DIR / "transfer_reddit_to_daic.csv", index=False
    )

    # Save confusion matrices
    save_transfer_confusions(
        daic_to_reddit_cms,
        title_prefix="DAIC→Reddit confusion (binary)",
        out_prefix="cm_transfer_daic_to_reddit",
        label_names=["Not Depressed / Control", "Depressed / Risk"],
    )
    save_transfer_confusions(
        reddit_to_daic_cms,
        title_prefix="Reddit→DAIC confusion (binary)",
        out_prefix="cm_transfer_reddit_to_daic",
        label_names=["Not Depressed / Control", "Depressed / Risk"],
    )

    # Compact key finding
    best_daic_to_reddit = daic_to_reddit_df.iloc[0]
    best_reddit_to_daic = reddit_to_daic_df.iloc[0]
    print(
        "\nKey finding (transfer): "
        f"Best DAIC→Reddit Weighted F1={best_daic_to_reddit['f1_weighted']:.3f} ({best_daic_to_reddit['model']}), "
        f"Best Reddit→DAIC Weighted F1={best_reddit_to_daic['f1_weighted']:.3f} ({best_reddit_to_daic['model']})."
    )

    print("\nSaved transfer artifacts to:", OUTPUT_DIR.resolve())
    print("- daic_embeddings.npy")
    print("- daic_labels_binary.npy")
    print("- transfer_daic_to_reddit.csv")
    print("- transfer_reddit_to_daic.csv")
    print("- cm_transfer_*.png")